In [ ]:
"""
NOTEBOOK 3: FEATURE EXTRACTOR PARA ML
======================================
Extrae features limpias para Anomaly Detection

INPUT: Log file (.txt)
OUTPUT: DataFrame con features para ML
"""

import pandas as pd
import numpy as np
import re
from datetime import datetime
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

LOG_FILE = "ruta/al/log.txt"  # ← CAMBIAR AQUÍ
OUTPUT_DIR = "outputs"

# Tags de ruido (filtrar)
NOISE_TAGS = [
    'WifiVendorHal', 'libc', 'gralloc4', 'MtkTvTime', 'TimeMsgProcess',
    'AuthPII', 'newrelic', 'e2prom', 'HISMW@hdmi', 'HISMW@hdmidata',
    'HotPlugDetectionAction', 'System.err', 'OomAdjuster', 'prk'
]

# Tags críticos
CRITICAL_TAGS = [
    'PqLink', 'ActivityManager', 'WindowManager', 'AppSearchManagerService',
    'FrameGenerator', '<MI3>'
]

# Patterns de eventos
CRASH_PATTERNS = {
    'has_crash': r'crash|CRASH',
    'has_anr': r'ANR|not responding',
    'has_fatal': r'fatal|FATAL',
    'has_restart': r'restart.*crashed'
}

# ============================================================================
# FUNCIONES
# ============================================================================

def parse_line(line: str) -> dict | None:
    if not line or line.startswith('-----'):
        return None
    
    try:
        pattern_a = r'^(\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s+(\d+)\s+(\d+)\s+([VDIWEF])\s+([^:]+):\s+(.+)$'
        match = re.match(pattern_a, line)
        
        if match:
            return {
                "date": match.group(1),
                "time": match.group(2),
                "pid": match.group(3),
                "tid": match.group(4),
                "level": match.group(5),
                "tag": match.group(6).strip(),
                "message": match.group(7)
            }
        
        pattern_b = r'^(\d{2}-\d{2})\s+(\d{2}:\d{2}:\d{2}\.\d{3})\s+([VDIWEF])/([^(]+)\(\s*(\d+)\):\s+(.+)$'
        match = re.match(pattern_b, line)
        
        if match:
            return {
                "date": match.group(1),
                "time": match.group(2),
                "level": match.group(3),
                "tag": match.group(4).strip(),
                "pid": match.group(5).strip(),
                "tid": match.group(5).strip(),
                "message": match.group(6)
            }
        
        return None
    except:
        return None


def time_rounded(parsed_line: str) -> str:
    time_minutes = datetime.strptime(parsed_line, "%H:%M:%S.%f")
    time_minutes = time_minutes.replace(second=0, microsecond=0).strftime("%H:%M")
    return time_minutes


def read_log(filepath: str):
    parsed_logs = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parsed = parse_line(line)
            if parsed:
                parsed["time"] = time_rounded(parsed["time"])
                parsed_logs.append(parsed)
    return parsed_logs


def extract_features(df):
    """Extrae features limpias por minuto"""
    
    # Filtrar ruido
    df_clean = df[~df['tag'].isin(NOISE_TAGS)].copy()
    
    # Agrupar por minuto
    by_minute = df_clean.groupby('time').agg({
        'level': [
            ('total_logs', 'count'),
            ('errors_critical', lambda x: (x == 'E').sum()),
            ('warnings_critical', lambda x: (x == 'W').sum()),
            ('fatals', lambda x: (x == 'F').sum())
        ]
    })
    
    by_minute.columns = ['_'.join(col).strip('_') for col in by_minute.columns]
    by_minute = by_minute.reset_index()
    
    # Error rates
    by_minute['error_rate'] = (by_minute['errors_critical'] / by_minute['total_logs'] * 100).fillna(0)
    by_minute['warning_rate'] = (by_minute['warnings_critical'] / by_minute['total_logs'] * 100).fillna(0)
    
    # Rolling windows
    by_minute['errors_rolling_3min'] = by_minute['errors_critical'].rolling(3, min_periods=1).sum()
    by_minute['errors_rolling_5min'] = by_minute['errors_critical'].rolling(5, min_periods=1).sum()
    
    # Deltas
    by_minute['error_delta'] = by_minute['errors_critical'].diff().fillna(0)
    
    # Volatilidad
    by_minute['error_std_5min'] = by_minute['errors_critical'].rolling(5, min_periods=1).std().fillna(0)
    
    return by_minute


def extract_global_features(df):
    """Features globales del log completo"""
    
    # Filtrar ruido
    df_clean = df[~df['tag'].isin(NOISE_TAGS)].copy()
    
    features = {}
    
    # Totales
    features['total_errors_critical'] = (df_clean['level'] == 'E').sum()
    features['total_warnings_critical'] = (df_clean['level'] == 'W').sum()
    features['total_fatals'] = (df_clean['level'] == 'F').sum()
    
    # Rates
    total = len(df_clean)
    features['error_rate_avg'] = features['total_errors_critical'] / total * 100 if total > 0 else 0
    
    # Tags críticos presentes
    for tag in CRITICAL_TAGS:
        features[f'has_{tag.replace("<", "").replace(">", "")}'] = int(tag in df_clean['tag'].values)
    
    # Eventos críticos
    for pattern_name, pattern in CRASH_PATTERNS.items():
        features[pattern_name] = int(df['message'].str.contains(pattern, case=False, regex=True, na=False).any())
    
    # Picos
    by_minute = df_clean.groupby('time').agg({
        'level': lambda x: (x == 'E').sum()
    })
    
    features['max_errors_per_min'] = by_minute['level'].max() if len(by_minute) > 0 else 0
    features['picos_count'] = int((by_minute['level'] > by_minute['level'].mean() + 2*by_minute['level'].std()).sum())
    
    return features


def main():
    print("="*80)
    print("🤖 FEATURE EXTRACTOR PARA ML")
    print("="*80)
    
    if not Path(LOG_FILE).exists():
        print(f"❌ Archivo no encontrado: {LOG_FILE}")
        return None, None
    
    print(f"\n📂 Procesando: {LOG_FILE}")
    
    # Parse
    parsed_logs = read_log(LOG_FILE)
    
    if len(parsed_logs) == 0:
        print("❌ No se pudo parsear el log")
        return None, None
    
    df = pd.DataFrame(parsed_logs)
    
    print(f"✅ Logs parseados: {len(df):,}")
    
    # Features por minuto
    print("\n🔍 Extrayendo features por minuto...")
    features_by_min = extract_features(df)
    
    print(f"✅ Features extraídas: {len(features_by_min)} minutos × {len(features_by_min.columns)} features")
    
    # Features globales
    print("\n🔍 Extrayendo features globales...")
    global_features = extract_global_features(df)
    
    print(f"✅ Features globales: {len(global_features)}")
    
    # Mostrar
    print("\n" + "="*80)
    print("📊 FEATURES BY MINUTE (primeras 10 filas)")
    print("="*80 + "\n")
    print(features_by_min.head(10).to_string())
    
    print("\n" + "="*80)
    print("📊 GLOBAL FEATURES")
    print("="*80 + "\n")
    for key, value in global_features.items():
        print(f"{key:40s} {value}")
    
    # Guardar
    Path(OUTPUT_DIR).mkdir(exist_ok=True)
    
    features_by_min.to_csv(f"{OUTPUT_DIR}/features_by_minute.csv", index=False)
    print(f"\n✅ Features guardadas: {OUTPUT_DIR}/features_by_minute.csv")
    
    pd.DataFrame([global_features]).to_csv(f"{OUTPUT_DIR}/features_global.csv", index=False)
    print(f"✅ Features globales guardadas: {OUTPUT_DIR}/features_global.csv")
    
    return features_by_min, global_features


if __name__ == "__main__":
    features_min, features_global = main()